In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # NB_08 — Nightly Score Refresh Job
# MAGIC **Scheduled Databricks Job · CDF-triggered incremental re-scoring**
# MAGIC
# MAGIC This is the "Nightly Score Refresh Job" box in the architecture diagram.
# MAGIC It is the feedback loop that makes XScore a living system, not a one-shot pipeline.
# MAGIC
# MAGIC What this notebook does:
# MAGIC - Reads CDF (Change Data Feed) from `silver.user_features`
# MAGIC - Detects only users whose features changed since last run
# MAGIC - Loads @Champion model from MLflow registry
# MAGIC - Re-scores only affected users (not all 50K)
# MAGIC - MERGEs new scores into `gold.credit_scores`
# MAGIC - Recomputes SHAP explanations for re-scored users
# MAGIC - Updates `gold.score_explanations`
# MAGIC - Logs run stats to `gold.refresh_log`
# MAGIC
# MAGIC **Scheduling:** Set as a Databricks Job, run nightly at 2 AM
# MAGIC **Depends on:** NB_05 and NB_07 complete
# MAGIC **Runtime:** 2–5 minutes (incremental — only changed users)

# COMMAND ----------
# MAGIC %md
# MAGIC ## Cell 1 — Setup

# COMMAND ----------

%pip install lightgbm shap --quiet

# COMMAND ----------

import mlflow
import mlflow.lightgbm
import lightgbm as lgb
import shap
import numpy as np
import pandas as pd
import pickle
import json
from datetime import datetime
from pyspark.sql import functions as F
from pyspark.sql.types import *
from delta.tables import DeltaTable

spark.sql("USE CATALOG xscore")
spark.conf.set("spark.sql.shuffle.partitions", "8")
mlflow.set_registry_uri("databricks-uc")

RUN_TIMESTAMP = datetime.now()
print(f"✓ Nightly refresh started: {RUN_TIMESTAMP.strftime('%Y-%m-%d %H:%M:%S')}")

# COMMAND ----------
# MAGIC %md
# MAGIC ## Cell 2 — Create refresh log table (first run only)

# COMMAND ----------

# MAGIC %sql
# MAGIC CREATE TABLE IF NOT EXISTS xscore.gold.refresh_log (
# MAGIC   run_timestamp       TIMESTAMP,
# MAGIC   last_processed_ver  BIGINT,
# MAGIC   new_version         BIGINT,
# MAGIC   changed_users       BIGINT,
# MAGIC   total_users_scored  BIGINT,
# MAGIC   champion_version    STRING,
# MAGIC   run_duration_secs   DOUBLE,
# MAGIC   status              STRING
# MAGIC )
# MAGIC USING DELTA
# MAGIC COMMENT 'Log of every nightly refresh run';

# COMMAND ----------
# MAGIC %md
# MAGIC ## Cell 3 — Get last processed CDF version

# COMMAND ----------

# The refresh log tracks which Delta version we last processed.
# We only read changes AFTER that version.

try:
    last_log = (spark.table("xscore.gold.refresh_log")
                .filter(F.col("status") == "SUCCESS")
                .orderBy("run_timestamp", ascending=False)
                .limit(1)
                .collect())
    if last_log:
        LAST_VERSION = int(last_log[0]["new_version"])
    else:
        LAST_VERSION = 0
except Exception:
    LAST_VERSION = 0

print(f"Last processed Silver version: {LAST_VERSION}")
print("Reading CDF changes since that version...")

# COMMAND ----------
# MAGIC %md
# MAGIC ## Cell 4 — Read CDF: which users changed?

# COMMAND ----------

# Read only rows that changed since last run
# CDF tracks: insert, update_preimage, update_postimage, delete
try:
    changes = (spark.read
               .format("delta")
               .option("readChangeFeed", "true")
               .option("startingVersion", LAST_VERSION)
               .table("xscore.silver.user_features"))

    # We only want the new/updated versions of changed rows
    changed_users_df = (changes
        .filter(F.col("_change_type").isin("insert", "update_postimage"))
        .select("user_id")
        .distinct())

    n_changed = changed_users_df.count()

    # Get current Silver version for logging
    from delta.tables import DeltaTable
    current_version = (DeltaTable.forName(spark, "xscore.silver.user_features")
                      .history(1)
                      .collect()[0]["version"])

    print(f"Changed users since version {LAST_VERSION}: {n_changed:,}")
    print(f"Current Silver version: {current_version}")

except Exception as e:
    print(f"CDF read error: {e}")
    print("Falling back to full re-score of all users...")
    changed_users_df = spark.table("xscore.silver.user_features").select("user_id")
    n_changed = changed_users_df.count()
    current_version = 0

# COMMAND ----------
# MAGIC %md
# MAGIC ## Cell 5 — Early exit if nothing changed

# COMMAND ----------

if n_changed == 0:
    print("No changes detected since last run.")
    print("Nothing to re-score. Logging and exiting.")

    # Log the no-op run
    log_row = spark.createDataFrame([{
        "run_timestamp"     : RUN_TIMESTAMP,
        "last_processed_ver": LAST_VERSION,
        "new_version"       : int(current_version),
        "changed_users"     : 0,
        "total_users_scored": 0,
        "champion_version"  : "N/A",
        "run_duration_secs" : 0.0,
        "status"            : "NO_CHANGES",
    }])
    log_row.write.format("delta").mode("append").saveAsTable("xscore.gold.refresh_log")

    dbutils.notebook.exit("NO_CHANGES")

print(f"Proceeding to re-score {n_changed:,} changed users")

# COMMAND ----------
# MAGIC %md
# MAGIC ## Cell 6 — Load feature contract and @Champion model

# COMMAND ----------

contract     = json.loads(dbutils.fs.head(
    "/Volumes/xscore/bronze/kaggle_raw/feature_contract.json"
))
FEATURE_COLS = contract["feature_cols"]

# Load best hyperparams from Gold table
best_params_row = (spark.table("xscore.gold.best_hyperparams")
                   .orderBy("run_timestamp", ascending=False)
                   .limit(1)
                   .collect())

if best_params_row:
    r = best_params_row[0]
    print(f"Using best hyperparams from: {r['run_timestamp']}")
    print(f"  AUC when tuned: {r['best_auc']:.4f}")
else:
    print("No hyperparams found — using @Champion from MLflow")

# Load @Champion model
champion_model = mlflow.lightgbm.load_model(
    "models:/xscore.gold.credit_scorer@Champion"
)
print(f"✓ @Champion model loaded")

# Load SHAP explainer directly from Volume
with open("/Volumes/xscore/bronze/kaggle_raw/lgbm_v3_explainer.pkl", "rb") as f:
    explainer = pickle.load(f)
print(f"✓ SHAP explainer loaded")

# COMMAND ----------
# MAGIC %md
# MAGIC ## Cell 7 — Load features for changed users only

# COMMAND ----------

# Get full feature rows for the changed users
changed_features = (spark.table("xscore.gold.credit_feature_store")
    .join(changed_users_df, on="user_id", how="inner")
    .select(["user_id", "segment"] + FEATURE_COLS)
    .fillna(0.0))

n_features = changed_features.count()
print(f"Feature rows for changed users: {n_features:,}")

# Convert to Pandas for LightGBM inference
changed_pd = changed_features.toPandas()
X_changed  = changed_pd[FEATURE_COLS]

# COMMAND ----------
# MAGIC %md
# MAGIC ## Cell 8 — Score changed users

# COMMAND ----------

start_score = datetime.now()

# Score
probs = champion_model.predict(X_changed)

changed_pd["default_probability"] = probs
changed_pd["xscore"] = (900 - probs * 900).astype(int)
changed_pd["score_band"] = pd.cut(
    changed_pd["xscore"],
    bins=[-1, 400, 600, 750, 901],
    labels=["Poor", "Fair", "Good", "Excellent"]
).astype(str)
changed_pd["score_timestamp"] = RUN_TIMESTAMP
changed_pd["model_version"]   = "nightly_refresh"
changed_pd["model_run_id"]    = "nightly_job"

# Convert to Spark
new_scores = spark.createDataFrame(
    changed_pd[["user_id","segment","xscore","score_band",
                "default_probability","score_timestamp",
                "model_version","model_run_id"]]
)

# MERGE into gold.credit_scores — only updates changed users
DeltaTable.forName(spark, "xscore.gold.credit_scores") \
    .alias("t").merge(new_scores.alias("s"), "t.user_id = s.user_id") \
    .whenMatchedUpdateAll() \
    .whenNotMatchedInsertAll() \
    .execute()

score_time = (datetime.now() - start_score).total_seconds()
print(f"✓ Re-scored {n_features:,} users in {score_time:.1f}s")
print(f"  Score distribution of re-scored users:")
display(new_scores.groupBy("score_band").count().orderBy("score_band"))

# COMMAND ----------
# MAGIC %md
# MAGIC ## Cell 9 — Recompute SHAP for changed users

# COMMAND ----------

print("Recomputing SHAP explanations for changed users...")

sv = explainer.shap_values(X_changed)
sv = sv[1] if isinstance(sv, list) else sv
sv_neg = -sv  # negate: positive = good for user

PILLAR_MAP = {
    "p1_bill_ontime_rate"   : "pillar_1_bill_payment",
    "p1_avg_days_late"      : "pillar_1_bill_payment",
    "p1_severe_late_rate"   : "pillar_1_bill_payment",
    "p1_bill_type_diversity": "pillar_1_bill_payment",
    "p1_payment_trend"      : "pillar_1_bill_payment",
    "p2_upi_txn_per_month"  : "pillar_2_upi_flow",
    "p2_avg_txn_amount"     : "pillar_2_upi_flow",
    "p2_txn_cv"             : "pillar_2_upi_flow",
    "p2_failure_rate"       : "pillar_2_upi_flow",
    "p2_merchant_diversity" : "pillar_2_upi_flow",
    "owns_land_d"           : "pillar_3_assets",
    "land_acres_capped"     : "pillar_3_assets",
    "owns_vehicle_d"        : "pillar_3_assets",
    "bank_vintage_capped"   : "pillar_3_assets",
    "has_fd_or_rd_d"        : "pillar_3_assets",
    "income_log"            : "pillar_4_income",
    "itr_filed_d"           : "pillar_4_income",
    "gst_registered_d"      : "pillar_4_income",
    "employment_capped"     : "pillar_4_income",
    "jan_dhan_active_d"     : "pillar_5_identity",
    "shg_member_d"          : "pillar_5_identity",
    "shg_months"            : "pillar_5_identity",
    "dbt_months"            : "pillar_5_identity",
    "svanidhi_repaid_d"     : "pillar_5_identity",
    "sim_tenure_months"     : "pillar_6_stability",
    "location_stability"    : "pillar_6_stability",
    "fraud_flag_d"          : "pillar_6_stability",
}

# Build SHAP DataFrame
shap_df = pd.DataFrame(sv_neg, columns=FEATURE_COLS)
shap_df["user_id"] = changed_pd["user_id"].values
shap_df["segment"] = changed_pd["segment"].values

# Per-pillar sums
pillars = set(PILLAR_MAP.values())
for p in pillars:
    feats = [f for f, pp in PILLAR_MAP.items() if pp == p]
    shap_df[f"shap_{p}_pts"] = (
        shap_df[feats].sum(axis=1).clip(-2, 2) / 2 * 150
    ).round(1)

# Top factors
READABLE = {
    "p1_bill_ontime_rate"   : "Bill payment consistency",
    "p1_avg_days_late"      : "Average payment delay",
    "p2_upi_txn_per_month"  : "UPI transaction volume",
    "p2_failure_rate"       : "UPI failure rate",
    "income_log"            : "Income level",
    "itr_filed_d"           : "ITR filing",
    "owns_land_d"           : "Land ownership",
    "bank_vintage_capped"   : "Banking tenure",
    "fraud_flag_d"          : "Fraud flag",
    "sim_tenure_months"     : "SIM tenure",
    "jan_dhan_active_d"     : "Jan Dhan activity",
    "shg_member_d"          : "SHG membership",
}

sv_matrix = shap_df[FEATURE_COLS].values
pos_list, neg_list, text_list = [], [], []

for i, row in enumerate(sv_matrix):
    s = pd.Series(dict(zip(FEATURE_COLS, row)))
    top_pos = s.nlargest(3)
    top_neg = s.nsmallest(3)
    pos_str = " | ".join([f"{READABLE.get(k,k)} (+{v:.2f})"
                          for k, v in top_pos.items() if v > 0])
    neg_str = " | ".join([f"{READABLE.get(k,k)} ({v:.2f})"
                          for k, v in top_neg.items() if v < 0])
    pos_list.append(pos_str or "No strong positive factors")
    neg_list.append(neg_str or "No strong negative factors")

    score = int(changed_pd.iloc[i]["xscore"])
    band  = str(changed_pd.iloc[i]["score_band"])
    top1  = list(top_pos.index)[0] if len(top_pos) else ""
    top1r = READABLE.get(top1, top1)
    top1n = list(top_neg.index)[0] if len(top_neg) else ""
    top1nr = READABLE.get(top1n, top1n)

    if pos_str:
        text_list.append(
            f"Score {score} ({band}). Strength: {top1r}. Opportunity: {top1nr}."
        )
    else:
        text_list.append(f"Score {score} ({band}). Build payment history to improve.")

shap_df["top_positive_factors"] = pos_list
shap_df["top_negative_factors"] = neg_list
shap_df["explanation_text"]     = text_list
shap_df["computed_at"]          = RUN_TIMESTAMP

# Select columns matching the score_explanations schema
pts_cols = [c for c in shap_df.columns if c.endswith("_pts")]
out_cols  = (["user_id","segment","explanation_text",
               "top_positive_factors","top_negative_factors"]
             + pts_cols + ["computed_at"])

new_explanations = spark.createDataFrame(shap_df[out_cols])

DeltaTable.forName(spark, "xscore.gold.score_explanations") \
    .alias("t").merge(new_explanations.alias("s"), "t.user_id = s.user_id") \
    .whenMatchedUpdateAll() \
    .whenNotMatchedInsertAll() \
    .execute()

print(f"✓ SHAP explanations updated for {n_features:,} users")

# COMMAND ----------
# MAGIC %md
# MAGIC ## Cell 10 — Log run to refresh_log

# COMMAND ----------

end_time  = datetime.now()
duration  = (end_time - RUN_TIMESTAMP).total_seconds()
total_n   = spark.table("xscore.gold.credit_scores").count()

log_row = spark.createDataFrame([{
    "run_timestamp"     : RUN_TIMESTAMP,
    "last_processed_ver": int(LAST_VERSION),
    "new_version"       : int(current_version),
    "changed_users"     : int(n_changed),
    "total_users_scored": int(total_n),
    "champion_version"  : "Champion",
    "run_duration_secs" : float(duration),
    "status"            : "SUCCESS",
}])

log_row.write.format("delta").mode("append").saveAsTable("xscore.gold.refresh_log")

print(f"✓ Run logged to xscore.gold.refresh_log")

# COMMAND ----------
# MAGIC %md
# MAGIC ## Cell 11 — Final summary

# COMMAND ----------

print("=" * 60)
print("  NB_08 NIGHTLY REFRESH — COMPLETE")
print("=" * 60)
print(f"  Run timestamp    : {RUN_TIMESTAMP.strftime('%Y-%m-%d %H:%M:%S')}")
print(f"  Changed users    : {n_changed:,}")
print(f"  Total in Gold    : {total_n:,}")
print(f"  Duration         : {duration:.1f}s")
print(f"  Silver version   : {LAST_VERSION} → {current_version}")
print()
print("  This notebook is scheduled as a Databricks Job.")
print("  To schedule:")
print("    Sidebar → Workflows → Create Job")
print("    → Add task → Notebook → select NB_08")
print("    → Schedule → Every day at 02:00 AM")
print("=" * 60)